In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Qwen3-4B — 4-Class Syllogism Classifier (Unsloth)
# FIXED VERSION — Single NUM_LAST_LAYERS parameter
#   • Set NUM_LAST_LAYERS = 8, 10, or 12
#   • LAYER_INDICES auto-generated as contiguous last N layers
#   • Confusion matrix: TEST DATA ONLY (matching paper style)
#   • Model saved as .pt file (not .bin)
#   • LIME with proper visualization
#   • CFG = original CFG_A hyperparameters (proven working)
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# >>>  USER: SET THIS ONE PARAMETER  <<<
# NUM_LAST_LAYERS = 8   # last 8 layers  → expect ~0.95
# NUM_LAST_LAYERS = 10  # last 10 layers → expect ~0.94
# NUM_LAST_LAYERS = 12  # last 12 layers → expect ~0.93
# ══════════════════════════════════════════════════════════════════════
NUM_LAST_LAYERS = 8  # <--- CHANGE THIS: 8, 10, or 12
# ══════════════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════
# CELL 1 — Install Dependencies
# ════════════════════════════════════════════════════════
import os, subprocess, sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth", "unsloth_zoo"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.51.0", "peft", "accelerate",
                "scikit-learn", "seaborn", "matplotlib", "openpyxl",
                "huggingface_hub", "lime"], check=True)
print("✓ All dependencies installed")

# ════════════════════════════════════════════════════════
# CELL 2 — HuggingFace Login
# ════════════════════════════════════════════════════════
from huggingface_hub import login
from getpass import getpass

def safe_login():
    raw = getpass("Enter HF token (hf_...): ")
    token = "".join(c for c in raw if 32 <= ord(c) <= 126).strip()
    if not token.startswith("hf_"):
        raise ValueError("Token must start with hf_")
    login(token=token, add_to_git_credential=False)
    print("✓ HuggingFace login OK")

safe_login()

# ════════════════════════════════════════════════════════
# CELL 3 — Imports & Global Constants
# ════════════════════════════════════════════════════════
import gc, json, random, warnings, time, shutil
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

from unsloth import FastLanguageModel
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, AutoTokenizer
from transformers.modeling_outputs import SequenceClassifierOutputWithPast
from torch.utils.data import Dataset as TorchDataset
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from lime.lime_text import LimeTextExplainer

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "unsloth/Qwen3-4B"
N_LAYERS = 36
HIDDEN = 2560
MAX_SEQ = 2048
NUM_LABELS = 4

LABEL2ID = {
    "valid and plausible": 0,
    "valid and implausible": 1,
    "invalid and plausible": 2,
    "invalid and implausible": 3,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
SHORT_LABELS = ["V+P", "V+!P", "!V+P", "!V+!P"]

# --- Auto-generate layer indices based on NUM_LAST_LAYERS ---
LAYER_INDICES = [-i for i in range(1, NUM_LAST_LAYERS + 1)]
print(f"Auto-generated LAYER_INDICES for last {NUM_LAST_LAYERS} layers: {LAYER_INDICES}")

# Use the curated file produced earlier. Upload it to /content first.
TRAIN_CSV = "/content/qwen3_syllogism_train_CURATED_AUG_NONLEAKY.csv"
TEST_CSV  = "/content/test_data_with_labels.csv"
MAX_LENGTH = 320
VAL_RATIO = 0.10

RUN_DATE = datetime.now().strftime("%Y%m%d_%H%M")
DATE_LABEL = datetime.now().strftime("%d %b %Y %H:%M")
BASE_OUTPUT = f"/content/qwen3_4b_{RUN_DATE}"
os.makedirs(BASE_OUTPUT, exist_ok=True)
DRIVE_BASE = f"/content/drive/MyDrive/SemEval2026/Qwen3_4B_Unsloth/{RUN_DATE}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Model   : {MODEL_NAME} ({N_LAYERS}L hidden={HIDDEN})")
print(f"Date    : {DATE_LABEL}")
print(f"Layers  : last {NUM_LAST_LAYERS} → {LAYER_INDICES}")
print("✓ Global config ready")

# ════════════════════════════════════════════════════════
# CELL 4 — Experiment Config (ORIGINAL CFG_A VALUES)
#         Only layer/feature params auto-generated from NUM_LAST_LAYERS
# ════════════════════════════════════════════════════════
CFG = dict(
    NAME=f"CFG_Last{NUM_LAST_LAYERS}_OrigLoRA_SavePT",

    # ---- LoRA: ORIGINAL proven values ----
    LORA_R=64,
    LORA_ALPHA=128,
    LORA_DROPOUT=0.0,        # 0.0 for Unsloth fast patching
    LORA_TARGETS=["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"],

    # ---- Layer pooling (auto-generated from NUM_LAST_LAYERS) ----
    LAYER_INDICES=LAYER_INDICES,
    POOL_MODE="weighted",

    # ---- Unfreeze: ORIGINAL values ----
    UNFREEZE_ATTN_NORM=True,
    UNFREEZE_LM_HEAD=True,
    UNFREEZE_LAST_N_BLOCKS=12,  # original proven value

    # ---- DenseResHead: ORIGINAL proven values ----
    HEAD_DROPOUT=0.05,
    GROWTH_RATE=256,
    NUM_DENSE_LAYERS=4,
    BOTTLENECK_FACTOR=4,

    LABEL_SMOOTHING=0.05,

    # ---- Training: ORIGINAL proven values ----
    LEARNING_RATE=1.5e-4,
    MAX_EPOCHS=6,
    PATIENCE=5,
    EARLY_STOP_THRESH=0.001,
    PER_DEVICE_BATCH=2,
    GRAD_ACCUM=8,
    WARMUP_RATIO=0.10,
    WEIGHT_DECAY=0.01,
    MAX_GRAD_NORM=1.0,
    LR_SCHEDULER="cosine_with_restarts",
    NUM_CYCLES=1,
    EVAL_STEPS=75,

    USE_TTA=True,
    TTA_VARIANTS=3,

    USE_LIME=True,
    LIME_N_SAMPLES=10,
    LIME_NUM_FEATURES=8,
    LIME_NUM_PERTURB=512,
)
print("✓ Config ready")
print(f"  NUM_LAST_LAYERS = {NUM_LAST_LAYERS}")
print(f"  Layer indices   : {CFG['LAYER_INDICES']}")
print(f"  LoRA r          : {CFG['LORA_R']}")
print(f"  Head growth     : {CFG['GROWTH_RATE']} × {CFG['NUM_DENSE_LAYERS']}")
print(f"  Max epochs      : {CFG['MAX_EPOCHS']}")
print(f"  Unfreeze blocks : {CFG['UNFREEZE_LAST_N_BLOCKS']}")

# ════════════════════════════════════════════════════════
# CELL 5 — DenseResHead (ORIGINAL architecture)
# ════════════════════════════════════════════════════════
class DenseResHead(nn.Module):
    def __init__(self, input_size: int, num_labels: int, growth_rate: int = 256,
                 num_dense_layers: int = 4, bottleneck_factor: int = 4, dropout: float = 0.05):
        super().__init__()
        bn_dim = growth_rate * bottleneck_factor
        self.bottleneck = nn.Sequential(
            nn.Linear(input_size, bn_dim),
            nn.LayerNorm(bn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.dense_layers = nn.ModuleList()
        self.dense_norms = nn.ModuleList()
        in_ch = bn_dim
        for _ in range(num_dense_layers):
            self.dense_layers.append(nn.Linear(in_ch, growth_rate))
            self.dense_norms.append(nn.LayerNorm(growth_rate))
            in_ch += growth_rate
        t_dim = max(growth_rate * 2, num_labels * 16)
        self.transition = nn.Sequential(
            nn.Linear(in_ch, t_dim),
            nn.LayerNorm(t_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        self.classifier = nn.Linear(t_dim, num_labels)
        self.drop = nn.Dropout(dropout)
        self.act = nn.GELU()
        self._init_weights()
        print(f"  [DenseResHead] {input_size} → bn{bn_dim} → dense×{num_dense_layers}(g={growth_rate}) → trans{t_dim} → {num_labels}")

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.bottleneck(x)
        acc = [h]
        for layer, norm in zip(self.dense_layers, self.dense_norms):
            new = self.drop(self.act(norm(layer(torch.cat(acc, dim=-1)))))
            acc.append(new)
        return self.classifier(self.transition(torch.cat(acc, dim=-1)))


# ════════════════════════════════════════════════════════
# CELL 6 — Qwen3 Classifier Wrapper
# ════════════════════════════════════════════════════════
class Qwen3_4B_Classifier(nn.Module):
    def __init__(self, base_model, hidden_size: int, num_labels: int,
                 layer_indices: list, pool_mode: str = "weighted",
                 head_dropout: float = 0.05, growth_rate: int = 256,
                 num_dense_layers: int = 4, bottleneck_factor: int = 4,
                 label_smoothing: float = 0.05):
        super().__init__()
        self.base = base_model
        self.config = base_model.config
        self.num_labels = num_labels
        self.layer_indices = layer_indices
        self.pool_mode = pool_mode
        self.label_smoothing = label_smoothing
        self.layer_weights = nn.Parameter(
            torch.ones(len(layer_indices), dtype=torch.float32) / len(layer_indices)
        )
        self.head = DenseResHead(hidden_size, num_labels, growth_rate,
                                 num_dense_layers, bottleneck_factor, head_dropout)

    def gradient_checkpointing_enable(self, **kwargs):
        if hasattr(self.base, "gradient_checkpointing_enable"):
            self.base.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        if hasattr(self.base, "gradient_checkpointing_disable"):
            self.base.gradient_checkpointing_disable()

    def _pool(self, hidden_states: tuple, attention_mask: torch.Tensor) -> torch.Tensor:
        total = len(hidden_states)
        abs_idxs = [
            max(1, min((total + li) if li < 0 else (li + 1), total - 1))
            for li in self.layer_indices
        ]
        B = hidden_states[0].shape[0]
        seq_lens = attention_mask.sum(1) - 1
        vecs = []
        for ai in abs_idxs:
            hs = hidden_states[ai]
            idx = seq_lens.clamp(0).unsqueeze(-1).unsqueeze(-1).expand(B, 1, hs.shape[-1])
            vecs.append(hs.gather(1, idx).squeeze(1))
        w = torch.softmax(self.layer_weights, dim=0)
        stacked = torch.stack(vecs, dim=0)
        return (stacked * w.view(-1, 1, 1)).sum(0)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                labels: torch.Tensor = None):
        out = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False,
        )
        pooled = self._pool(out.hidden_states, attention_mask)
        logits = self.head(pooled.float())
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(label_smoothing=self.label_smoothing)(
                logits, labels.to(logits.device)
            )
        # CRITICAL: Do NOT return hidden_states to avoid OOM in Trainer eval
        return SequenceClassifierOutputWithPast(loss=loss, logits=logits)


# ════════════════════════════════════════════════════════
# CELL 7 — PtBinTrainer (memory-safe, saves .pt files)
# ════════════════════════════════════════════════════════
class PtBinTrainer(Trainer):
    def _get_processor(self):
        return getattr(self, "processing_class", None) or getattr(self, "tokenizer", None)

    def save_model(self, output_dir: str, state_dict=None, **kwargs):
        """Save model as .pt file instead of .bin
        Accepts **kwargs (e.g. _internal_call from newer transformers) to avoid TypeError.
        """
        os.makedirs(output_dir, exist_ok=True)
        if state_dict is None:
            state_dict = self.model.state_dict()
        # Save as .pt file
        pt_path = os.path.join(output_dir, "pytorch_model.pt")
        torch.save(state_dict, pt_path)
        print(f"  ✓ Model state_dict saved → {pt_path}")

        # Save config
        if hasattr(self.model, "config") and self.model.config is not None:
            try:
                self.model.config.save_pretrained(output_dir)
            except Exception:
                pass

        # Save tokenizer/processor
        proc = self._get_processor()
        if proc is not None:
            try:
                proc.save_pretrained(output_dir)
            except Exception:
                pass

        # Save classifier metadata
        cfg = getattr(self.model, "layer_indices", None)
        meta = {
            "note": "Qwen3_4B_Classifier wrapped model. Load by constructing classifier then load_state_dict(torch.load('pytorch_model.pt', map_location='cpu')).",
            "num_labels": self.model.num_labels if hasattr(self.model, "num_labels") else NUM_LABELS,
            "layer_indices": cfg if cfg else LAYER_INDICES,
            "label2id": LABEL2ID,
            "id2label": ID2LABEL,
            "pool_mode": "weighted",
        }
        with open(os.path.join(output_dir, "classifier_metadata.json"), "w") as f:
            json.dump(meta, f, indent=2)

    def _load_best_model(self):
        ckpt = getattr(self.state, "best_model_checkpoint", None)
        if ckpt:
            pt_candidate = os.path.join(ckpt, "pytorch_model.pt")
            bin_candidate = os.path.join(ckpt, "pytorch_model.bin")
            if os.path.isfile(pt_candidate):
                print(f"  ← Loading best checkpoint .pt: {pt_candidate}")
                sd = torch.load(pt_candidate, map_location="cpu")
                self.model.load_state_dict(sd, strict=False)
                return
            elif os.path.isfile(bin_candidate):
                print(f"  ← Loading best checkpoint .bin: {bin_candidate}")
                sd = torch.load(bin_candidate, map_location="cpu")
                self.model.load_state_dict(sd, strict=False)
                return
        super()._load_best_model()

    def compute_loss(self, model, inputs, return_outputs: bool = False,
                     num_items_in_batch=None):
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=inputs.get("labels"),
        )
        return (outputs.loss, outputs) if return_outputs else outputs.loss

    def prediction_step(self, model, inputs, prediction_loss_only: bool,
                        ignore_keys=None):
        """OOM-safe: only returns loss/logits/labels, never hidden_states."""
        inputs = self._prepare_inputs(inputs)
        labels = inputs.get("labels", None)
        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                labels=labels,
            )
        loss = outputs.loss.detach() if labels is not None else None
        logits = outputs.logits.detach()
        if prediction_loss_only:
            return (loss, None, None)
        if labels is not None:
            labels = labels.detach()
        return (loss, logits, labels)


# ════════════════════════════════════════════════════════
# CELL 8 — Prompts and TTA Variants
# ════════════════════════════════════════════════════════
_SYS_BASE = (
    "/no_think\n"
    "Task: classify a syllogism on TWO independent axes.\n"
    "[V]alid = the logical inference is sound (ignore real-world facts).\n"
    "[P]lausible = the stated facts are true (ignore logical structure).\n"
    "Choose exactly one label:\n"
    "valid and plausible | valid and implausible | invalid and plausible | invalid and implausible\n"
    "Output the label ONLY — no explanation."
)
_SHOTS = (
    "Example 1 [V=YES, P=YES]:\n"
    "All mammals breathe air. Dogs are mammals. Therefore dogs breathe air.\n"
    "→ valid and plausible\n"
    "Example 2 [V=YES, P=NO]:\n"
    "All rocks eat food. This pebble is a rock. Therefore this pebble eats.\n"
    "→ valid and implausible\n"
    "Example 3 [V=NO, P=YES]:\n"
    "Most birds can fly. Eagles can fly. Therefore eagles are birds.\n"
    "→ invalid and plausible\n"
    "Example 4 [V=NO, P=NO]:\n"
    "Fire is cold. Ice melts in cold. Therefore ice melts in fire.\n"
    "→ invalid and implausible"
)

def build_prompt_v1(syl: str) -> str:
    return (f"<|im_start|>system\n{_SYS_BASE}<|im_end|>\n"
            f"<|im_start|>user\n{_SHOTS}\n\nNow classify: {syl.strip()}<|im_end|>\n"
            f"<|im_start|>assistant\nLabel:")

def build_prompt_v2(syl: str) -> str:
    sys2 = (
        "/no_think\nYou are a logic evaluator. Assess VALIDITY and PLAUSIBILITY independently.\n"
        "Return exactly one label: valid and plausible | valid and implausible | invalid and plausible | invalid and implausible.\n"
        "Label only."
    )
    return (f"<|im_start|>system\n{sys2}<|im_end|>\n"
            f"<|im_start|>user\n{_SHOTS}\n\nAnalyse: {syl.strip()}<|im_end|>\n"
            f"<|im_start|>assistant\nLabel:")

def build_prompt_v3(syl: str) -> str:
    sys3 = (
        "/no_think\nDual-axis syllogism labelling. VALID=logic sound. PLAUSIBLE=facts true.\n"
        "Labels: valid and plausible | valid and implausible | invalid and plausible | invalid and implausible. Output label only."
    )
    return (f"<|im_start|>system\n{sys3}<|im_end|>\n"
            f"<|im_start|>user\nReference examples:\n{_SHOTS}\n\nLabel this: {syl.strip()}<|im_end|>\n"
            f"<|im_start|>assistant\nLabel:")

def build_prompt(syl: str) -> str:
    return build_prompt_v1(syl)

PROMPT_VARIANTS = [build_prompt_v1, build_prompt_v2, build_prompt_v3]
print(f"✓ {len(PROMPT_VARIANTS)} prompt variants ready")


# ════════════════════════════════════════════════════════
# CELL 9 — Dataset and Metrics
# ════════════════════════════════════════════════════════
class SyllogismDataset(TorchDataset):
    def __init__(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: list):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx: int):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


def build_dataset(df: pd.DataFrame, tokenizer, max_len: int) -> SyllogismDataset:
    enc = tokenizer(
        [build_prompt(s) for s in df["syllogism"].tolist()],
        max_length=max_len, padding="max_length", truncation=True, return_tensors="pt",
    )
    return SyllogismDataset(enc["input_ids"], enc["attention_mask"], df["label_id"].tolist())


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "accuracy": accuracy_score(labels, preds),
    }


def full_report(y_true, y_pred, title=""):
    mac = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mic = f1_score(y_true, y_pred, average="micro", zero_division=0)
    wtd = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    pre = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    lab = list(range(NUM_LABELS))
    pc_p = precision_score(y_true, y_pred, average=None, labels=lab, zero_division=0)
    pc_r = recall_score(y_true, y_pred, average=None, labels=lab, zero_division=0)
    pc_f = f1_score(y_true, y_pred, average=None, labels=lab, zero_division=0)
    print("\n" + "═" * 65 + f"\n  {title}\n" + "═" * 65)
    print(f"  Macro F1={mac:.4f} | Micro F1={mic:.4f} | Weighted F1={wtd:.4f} | Acc={acc:.4f}")
    for i in range(NUM_LABELS):
        n = sum(1 for y in y_true if y == i)
        print(f"    [{i}] {ID2LABEL[i]:<28} P={pc_p[i]:.3f} R={pc_r[i]:.3f} F1={pc_f[i]:.3f} N={n}")
    return dict(macro_f1=mac, micro_f1=mic, weighted_f1=wtd, precision=pre, recall=rec,
                accuracy=acc, per_class_prec=pc_p.tolist(), per_class_rec=pc_r.tolist(),
                per_class_f1=pc_f.tolist())


# ════════════════════════════════════════════════════════
# CELL 10 — Plot, Predict, LIME, Excel
# ════════════════════════════════════════════════════════

def plot_test_cm(test_true, test_pred, cfg_name, test_f1, save_path):
    """Single 4×4 confusion matrix — TEST DATA ONLY (like Fig 3 in paper)."""
    cm = confusion_matrix(test_true, test_pred, labels=list(range(NUM_LABELS)))
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=SHORT_LABELS, yticklabels=SHORT_LABELS,
                ax=ax, cbar=False, annot_kws={"size": 14, "weight": "bold"})
    ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
    ax.set_ylabel("True Label", fontsize=12, fontweight="bold")
    ax.set_title(
        f"Test Confusion Matrix — Last {NUM_LAST_LAYERS} Layers\n"
        f"Macro F1={test_f1:.4f} | {DATE_LABEL}",
        fontsize=11, fontweight="bold"
    )
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"  Test CM → {save_path}")


@torch.no_grad()
def predict_single(model, tokenizer, texts, prompt_fn, batch_size: int = 4):
    device = next(model.parameters()).device
    model.eval()
    all_probs = []
    for s in range(0, len(texts), batch_size):
        batch = texts[s:s + batch_size]
        enc = tokenizer(
            [prompt_fn(t) for t in batch],
            max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors="pt",
        )
        out = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        )
        probs = torch.softmax(out.logits.float(), dim=-1).cpu().numpy()
        all_probs.extend(probs.tolist())
        del enc, out, probs
        torch.cuda.empty_cache()
    return np.array(all_probs)


def predict(model, tokenizer, texts, batch_size: int = 4,
            use_tta: bool = False, n_variants: int = 3):
    if not use_tta:
        probs = predict_single(model, tokenizer, texts, PROMPT_VARIANTS[0], batch_size)
    else:
        n = min(n_variants, len(PROMPT_VARIANTS))
        print(f"  TTA: soft-voting over {n} prompt variants …")
        stack = np.stack([
            predict_single(model, tokenizer, texts, PROMPT_VARIANTS[i], batch_size)
            for i in range(n)
        ], axis=0)
        probs = stack.mean(0)
    return probs.argmax(-1).tolist(), probs.tolist()


def run_lime_explanations(model, tokenizer, test_texts, test_labels,
                          test_preds, cfg, save_dir):
    if not cfg["USE_LIME"]:
        return []
    os.makedirs(save_dir, exist_ok=True)
    device = next(model.parameters()).device
    model.eval()

    def predictor(texts):
        probs = []
        for s in range(0, len(texts), 4):
            batch = texts[s:s + 4]
            enc = tokenizer(
                [build_prompt(t) for t in batch],
                max_length=MAX_LENGTH, padding="max_length", truncation=True,
                return_tensors="pt",
            )
            with torch.no_grad():
                out = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                )
            probs.extend(torch.softmax(out.logits.float(), -1).cpu().numpy().tolist())
            del enc, out
        return np.array(probs)

    explainer = LimeTextExplainer(class_names=list(ID2LABEL.values()))
    n_explain = min(cfg["LIME_N_SAMPLES"], len(test_texts))
    indices = random.sample(range(len(test_texts)), n_explain)
    rows = []
    print(f"\n  LIME: explaining {n_explain} examples …")
    for rank, idx in enumerate(indices, 1):
        try:
            exp = explainer.explain_instance(
                test_texts[idx], predictor,
                num_features=cfg["LIME_NUM_FEATURES"],
                num_samples=cfg["LIME_NUM_PERTURB"],
                labels=list(range(NUM_LABELS)),
            )
            pred_label = test_preds[idx]
            for feat, weight in exp.as_list(label=pred_label):
                rows.append(dict(
                    RunDate=DATE_LABEL, ExampleIndex=idx, ExampleRank=rank,
                    TrueLabel=ID2LABEL[test_labels[idx]],
                    PredLabel=ID2LABEL[pred_label],
                    Correct=test_labels[idx] == pred_label,
                    ExplainedClass=ID2LABEL[pred_label],
                    Feature=feat, LIMEWeight=round(float(weight), 6),
                    Syllogism=test_texts[idx][:240],
                ))
            # Save LIME figure
            fig = exp.as_pyplot_figure(label=pred_label)
            fig.suptitle(
                f"LIME — Last {NUM_LAST_LAYERS} Layers\n"
                f"idx={idx} True={ID2LABEL[test_labels[idx]]} "
                f"Pred={ID2LABEL[pred_label]}",
                fontsize=10, fontweight="bold",
            )
            fig.tight_layout()
            fig.savefig(
                os.path.join(save_dir, f"lime_idx{idx:04d}.png"),
                dpi=120, bbox_inches="tight",
            )
            plt.close(fig)
        except Exception as e:
            print(f"    LIME failed for idx={idx}: {e}")
            plt.close("all")
    print(f"  LIME done: {len(rows)} feature rows")
    return rows


# --- Excel helpers ---
def _hdr(cell, bg="1F4E79"):
    cell.font = Font(bold=True, color="FFFFFF", size=11)
    cell.fill = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)


def _bdr():
    s = Side(style="thin", color="BFBFBF")
    return Border(left=s, right=s, top=s, bottom=s)


def _autowidth(ws, cols, rows, min_w=10, max_w=60):
    for ci, c in enumerate(cols, 1):
        mx = max(
            len(str(c)),
            max((len(str(r.get(c, ""))) for r in rows), default=0),
        )
        ws.column_dimensions[get_column_letter(ci)].width = min(max(mx + 3, min_w), max_w)


def _color_f1(cell, v):
    if isinstance(v, (float, int)):
        if float(v) >= 0.95:
            cell.fill = PatternFill("solid", fgColor="C6EFCE")
        elif float(v) >= 0.90:
            cell.fill = PatternFill("solid", fgColor="FFEB9C")
        else:
            cell.fill = PatternFill("solid", fgColor="FFC7CE")


def save_excel(summary_rows, pc_rows, lw_rows, lime_rows, arch_rows, path):
    wb = openpyxl.Workbook()
    sheet_specs = [
        ("Summary", summary_rows),
        ("PerClass", pc_rows),
        ("LayerWeights", lw_rows),
        ("LIME_Explanations", lime_rows),
        ("Architecture", arch_rows),
    ]

    # Sheet 1: Summary
    ws = wb.active
    ws.title = "Summary"
    cols = list(summary_rows[0].keys()) if summary_rows else ["NoData"]
    for ci, c in enumerate(cols, 1):
        cell = ws.cell(1, ci, c)
        _hdr(cell)
        cell.border = _bdr()
    for ri, row in enumerate(summary_rows, 2):
        for ci, c in enumerate(cols, 1):
            cell = ws.cell(ri, ci, row.get(c, ""))
            cell.border = _bdr()
            if c == "MacroF1":
                _color_f1(cell, row.get(c))
    _autowidth(ws, cols, summary_rows)
    ws.freeze_panes = "A2"

    # Config sheet
    ws = wb.create_sheet("Config")
    for ci, h in enumerate(["Section", "Parameter", "Value"], 1):
        cell = ws.cell(1, ci, h)
        _hdr(cell, "4472C4")
        cell.border = _bdr()
    config_rows = []
    for k, v in CFG.items():
        config_rows.append({"Section": "CFG", "Parameter": k, "Value": str(v)})
    extras = [
        ("Model/Data", "MODEL_NAME", MODEL_NAME),
        ("Model/Data", "N_LAYERS", N_LAYERS),
        ("Model/Data", "HIDDEN", HIDDEN),
        ("Model/Data", "NUM_LAST_LAYERS", NUM_LAST_LAYERS),
        ("Model/Data", "MAX_LENGTH", MAX_LENGTH),
        ("Model/Data", "TRAIN_CSV", TRAIN_CSV),
        ("Model/Data", "TEST_CSV", TEST_CSV),
    ]
    for sec, pk, pv in extras:
        config_rows.append({"Section": sec, "Parameter": pk, "Value": str(pv)})
    for ri, row in enumerate(config_rows, 2):
        ws.cell(ri, 1, row["Section"])
        ws.cell(ri, 2, row["Parameter"])
        ws.cell(ri, 3, row["Value"])
        for ci in range(1, 4):
            ws.cell(ri, ci).border = _bdr()
    ws.column_dimensions["A"].width = 18
    ws.column_dimensions["B"].width = 32
    ws.column_dimensions["C"].width = 80
    ws.freeze_panes = "A2"

    # Other sheets
    for title, rows in sheet_specs[1:]:
        ws = wb.create_sheet(title)
        cols = list(rows[0].keys()) if rows else ["NoData"]
        for ci, c in enumerate(cols, 1):
            cell = ws.cell(1, ci, c)
            _hdr(cell)
            cell.border = _bdr()
        for ri, row in enumerate(rows, 2):
            for ci, c in enumerate(cols, 1):
                cell = ws.cell(ri, ci, row.get(c, ""))
                cell.border = _bdr()
                if c in ["F1", "MacroF1"]:
                    _color_f1(cell, row.get(c))
        _autowidth(ws, cols, rows)
        ws.freeze_panes = "A2"

    wb.save(path)
    print(f"  Excel → {path}")


# ════════════════════════════════════════════════════════
# CELL 11 — Data Loading + Audit
# ════════════════════════════════════════════════════════
def load_data(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing CSV: {path}")
    df = pd.read_csv(path)
    if "gold_label" in df.columns and "label" not in df.columns:
        df = df.rename(columns={"gold_label": "label"})
    needed = {"id", "syllogism", "label"}
    if not needed.issubset(df.columns):
        raise ValueError(f"{path} must contain columns {needed}; got {df.columns.tolist()}")
    bad = sorted(set(df["label"]) - set(LABEL2ID))
    if bad:
        raise ValueError(f"Unknown labels in {path}: {bad}")
    df["label_id"] = df["label"].map(LABEL2ID).astype(int)
    return df


train_df = load_data(TRAIN_CSV)
test_df = load_data(TEST_CSV)
if set(train_df["syllogism"]) & set(test_df["syllogism"]):
    raise ValueError("LEAKAGE: shared syllogisms in train/test")
if set(train_df["id"]) & set(test_df["id"]):
    raise ValueError("LEAKAGE: shared IDs in train/test")
print(f"✓ Leakage audit passed | Train={len(train_df)} | Test={len(test_df)}")

train_split, val_split = train_test_split(
    train_df, test_size=VAL_RATIO, random_state=SEED, stratify=train_df["label_id"]
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
print(f"✓ Split | Train={len(train_split)} | Val={len(val_split)} | Test={len(test_df)}")

# Token length audit
_tmp = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
counts = [len(_tmp.encode(build_prompt(s))) for s in pd.concat([train_df, test_df])["syllogism"].tolist()]
worst = max(counts)
print(f"✓ Prompt worst-case: {worst} tokens (limit={MAX_LENGTH})")
assert worst < MAX_LENGTH, f"Prompt too long: {worst}"
del _tmp
gc.collect()
print("✓ Token audit passed")


# ════════════════════════════════════════════════════════
# CELL 12 — Model Loader
# ════════════════════════════════════════════════════════
def load_unsloth_model(cfg):
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Loading {MODEL_NAME} in NF4 4-bit via Unsloth …")
    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ,
        dtype=None,
        load_in_4bit=True,
        trust_remote_code=True,
    )
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    base = FastLanguageModel.get_peft_model(
        base,
        r=cfg["LORA_R"],
        lora_alpha=cfg["LORA_ALPHA"],
        lora_dropout=cfg["LORA_DROPOUT"],
        target_modules=cfg["LORA_TARGETS"],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )

    n_layers = base.config.num_hidden_layers
    hs = base.config.hidden_size
    total_hs = n_layers + 1
    print(f"  hidden={hs} n_layers={n_layers} total_hs={total_hs}")
    for li in cfg["LAYER_INDICES"]:
        ai = total_hs + li if li < 0 else li + 1
        assert 1 <= ai <= n_layers, f"Layer index {li} → abs={ai} out of [1,{n_layers}]"
        print(f"  li={li:>4} → hidden_states[{ai:>2}] (Layer {n_layers + li if li < 0 else li})")

    if cfg["UNFREEZE_ATTN_NORM"]:
        first_unfreeze = n_layers - cfg["UNFREEZE_LAST_N_BLOCKS"]
        norm_params = 0
        for name, param in base.named_parameters():
            if "layernorm" in name.lower() or "layer_norm" in name.lower():
                parts = name.split(".")
                layer_i = None
                for j, p in enumerate(parts):
                    if p == "layers" and j + 1 < len(parts):
                        try:
                            layer_i = int(parts[j + 1])
                        except ValueError:
                            pass
                if layer_i is not None and layer_i >= first_unfreeze:
                    param.requires_grad = True
                    norm_params += param.numel()
        print(f"  ✓ Unfrozen attention norms in layers [{first_unfreeze}–{n_layers - 1}] ({norm_params:,} params)")

    if cfg["UNFREEZE_LM_HEAD"]:
        lm_params = 0
        for name, param in base.named_parameters():
            if "lm_head" in name:
                param.requires_grad = True
                lm_params += param.numel()
        print(f"  ✓ Unfrozen lm_head ({lm_params:,} params)")

    model = Qwen3_4B_Classifier(
        base_model=base,
        hidden_size=hs,
        num_labels=NUM_LABELS,
        layer_indices=cfg["LAYER_INDICES"],
        pool_mode=cfg["POOL_MODE"],
        head_dropout=cfg["HEAD_DROPOUT"],
        growth_rate=cfg["GROWTH_RATE"],
        num_dense_layers=cfg["NUM_DENSE_LAYERS"],
        bottleneck_factor=cfg["BOTTLENECK_FACTOR"],
        label_smoothing=cfg["LABEL_SMOOTHING"],
    )
    for name, p in model.named_parameters():
        if "head." in name or "layer_weights" in name:
            p.requires_grad = True

    total = sum(p.numel() for p in model.parameters())
    trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable: {trained:,} / {total:,} ({100 * trained / total:.2f}%)")

    arch_rows = []
    for name, p in model.named_parameters():
        note = "QLoRA adapter" if "lora_" in name else \
               "Unfrozen norm" if ("layernorm" in name.lower() and p.requires_grad) else \
               "lm_head" if "lm_head" in name else \
               "Frozen base" if not p.requires_grad else "Head / pool"
        arch_rows.append(dict(
            ParameterName=name, Shape=str(list(p.shape)), NumParams=p.numel(),
            Trainable=bool(p.requires_grad),
            Module=".".join(name.split(".")[:3]), Note=note,
        ))
    print(f"  Architecture table: {len(arch_rows)} parameter entries")
    return model, tokenizer, arch_rows


# ════════════════════════════════════════════════════════
# CELL 13 — Mount Drive
# ════════════════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"✓ Drive mounted → {DRIVE_BASE}")


# ════════════════════════════════════════════════════════
# CELL 14 — Main Run Function
# ════════════════════════════════════════════════════════
def run(cfg):
    out_dir = f"{BASE_OUTPUT}/{cfg['NAME']}"
    drv_dir = f"{DRIVE_BASE}/{cfg['NAME']}"
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(drv_dir, exist_ok=True)

    print("\n" + "═" * 70)
    print(f"  CONFIG: {cfg['NAME']}")
    print(f"  NUM_LAST_LAYERS = {NUM_LAST_LAYERS}")
    print(f"  Layer indices   = {cfg['LAYER_INDICES']}")
    print(f"  LoRA r={cfg['LORA_R']} alpha={cfg['LORA_ALPHA']}")
    print(f"  Epochs={cfg['MAX_EPOCHS']} | Unfreeze={cfg['UNFREEZE_LAST_N_BLOCKS']} blocks")
    print("═" * 70)

    model = tok = trainer = train_ds = val_ds = None
    try:
        print("\n[1/9] Loading model …")
        model, tok, arch_rows = load_unsloth_model(cfg)
        print(f"  VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

        print("[2/9] Tokenising datasets …")
        train_ds = build_dataset(train_split, tok, MAX_LENGTH)
        val_ds = build_dataset(val_split, tok, MAX_LENGTH)
        print(f"  train={len(train_ds)} val={len(val_ds)} test={len(test_df)}")

        spe = max(len(train_ds) // (cfg["PER_DEVICE_BATCH"] * cfg["GRAD_ACCUM"]), 1)
        total_steps = spe * cfg["MAX_EPOCHS"]
        warmup_steps = max(int(total_steps * cfg["WARMUP_RATIO"]), 5)
        print(f"[3/9] spe={spe} total_steps≈{total_steps} warmup={warmup_steps}")

        args = TrainingArguments(
            output_dir=out_dir,
            num_train_epochs=cfg["MAX_EPOCHS"],
            per_device_train_batch_size=cfg["PER_DEVICE_BATCH"],
            per_device_eval_batch_size=cfg["PER_DEVICE_BATCH"],
            gradient_accumulation_steps=cfg["GRAD_ACCUM"],
            eval_accumulation_steps=1,
            learning_rate=cfg["LEARNING_RATE"],
            lr_scheduler_type=cfg["LR_SCHEDULER"],
            lr_scheduler_kwargs={"num_cycles": cfg["NUM_CYCLES"]},
            warmup_steps=warmup_steps,
            optim="adamw_8bit",
            weight_decay=cfg["WEIGHT_DECAY"],
            max_grad_norm=cfg["MAX_GRAD_NORM"],
            label_smoothing_factor=0.0,
            fp16=True,
            bf16=False,
            eval_strategy="steps",
            eval_steps=cfg["EVAL_STEPS"],
            save_strategy="steps",
            save_steps=cfg["EVAL_STEPS"],
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            load_best_model_at_end=True,
            save_total_limit=2,
            gradient_checkpointing=True,
            dataloader_num_workers=0,
            logging_steps=10,
            report_to="none",
            seed=SEED,
        )
        trainer = PtBinTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics,
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=cfg["PATIENCE"],
                    early_stopping_threshold=cfg["EARLY_STOP_THRESH"],
                )
            ],
        )

        print("[4/9] Training …")
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        tr_res = trainer.train(ignore_keys_for_eval=[
            "hidden_states", "past_key_values", "attentions",
        ])
        elapsed = time.time() - t0
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        train_loss = float(tr_res.training_loss)
        print(f"  Loss={train_loss:.4f} Time={elapsed / 60:.1f}min PeakVRAM={peak_gb:.2f}GB")

        print("[5/9] Validation inference …")
        vp, vc = predict(model, tok, val_split["syllogism"].tolist(),
                         batch_size=4, use_tta=cfg["USE_TTA"],
                         n_variants=cfg["TTA_VARIANTS"])
        vm = full_report(val_split["label_id"].tolist(), vp,
                         f"{cfg['NAME']} — VALIDATION [{DATE_LABEL}]")

        print("[6/9] Test inference …")
        tp, tc = predict(model, tok, test_df["syllogism"].tolist(),
                         batch_size=4, use_tta=cfg["USE_TTA"],
                         n_variants=cfg["TTA_VARIANTS"])
        tm = full_report(test_df["label_id"].tolist(), tp,
                         f"{cfg['NAME']} — TEST [{DATE_LABEL}]")

        print("[7/9] Confusion matrix (TEST ONLY) …")
        cm_path = f"{out_dir}/confusion_matrix_test_{RUN_DATE}.png"
        plot_test_cm(
            test_df["label_id"].tolist(), tp,
            cfg_name=cfg["NAME"], test_f1=tm["macro_f1"],
            save_path=cm_path,
        )

        print("[8/9] LIME explanations …")
        lime_dir = f"{out_dir}/lime_figures"
        lime_rows = run_lime_explanations(
            model, tok, test_df["syllogism"].tolist(),
            test_df["label_id"].tolist(), tp, cfg, lime_dir,
        )

        print("[9/9] Saving artefacts …")
        # Predictions CSV
        res_df = test_df[["id", "syllogism", "label"]].copy()
        res_df["pred"] = [ID2LABEL[p] for p in tp]
        res_df["correct"] = res_df["label"] == res_df["pred"]
        res_df["run_date"] = DATE_LABEL
        ca = np.array(tc)
        for i, lbl in ID2LABEL.items():
            res_df[f"conf_{lbl.replace(' ', '_')}"] = ca[:, i]
        csv_path = f"{out_dir}/predictions_test_{RUN_DATE}.csv"
        res_df.to_csv(csv_path, index=False)
        print(f"  Test correct: {res_df.correct.sum()}/{len(res_df)} "
              f"({100 * res_df.correct.mean():.1f}%)")

        # Layer weights
        lw_rows = []
        lw_vals = torch.softmax(model.layer_weights.detach().cpu().float(), 0).numpy()
        n_l = model.base.config.num_hidden_layers
        tot_hs = n_l + 1
        for li, w in zip(cfg["LAYER_INDICES"], lw_vals):
            ai = tot_hs + li if li < 0 else li + 1
            lw_rows.append(dict(
                RunDate=DATE_LABEL, Config=cfg["NAME"],
                RelLayerIdx=li, AbsLayerIdx=ai,
                SoftmaxWeight=round(float(w), 6),
                WeightPct=f"{100 * w:.2f}%",
                LayerName=f"L{n_l + li if li < 0 else li}",
            ))
        np.save(f"{out_dir}/layer_weights_{RUN_DATE}.npy", lw_vals)

        # Build summary rows
        hp = dict(
            RunDate=DATE_LABEL, Config=cfg["NAME"],
            Model=MODEL_NAME, NLayers=N_LAYERS, Hidden=HIDDEN,
            NumLastLayers=NUM_LAST_LAYERS,
            TrainLoss=round(train_loss, 5), PeakVRAM_GB=round(peak_gb, 2),
            Runtime_min=round(elapsed / 60, 1),
            LoRA_r=cfg["LORA_R"], LoRA_alpha=cfg["LORA_ALPHA"],
            LoRA_dropout=cfg["LORA_DROPOUT"],
            LayerIndices=str(cfg["LAYER_INDICES"]),
            Epochs=cfg["MAX_EPOCHS"],
            EffBatch=cfg["PER_DEVICE_BATCH"] * cfg["GRAD_ACCUM"],
            UseTTA=cfg["USE_TTA"], TTA_Variants=cfg["TTA_VARIANTS"],
        )

        summary_rows, pc_rows = [], []
        for split_name, met, y_true in [
            ("Validation", vm, val_split["label_id"].tolist()),
            ("Test", tm, test_df["label_id"].tolist()),
        ]:
            summary_rows.append(dict(
                **hp, Split=split_name,
                MacroF1=round(met["macro_f1"], 4),
                MicroF1=round(met["micro_f1"], 4),
                WtdF1=round(met["weighted_f1"], 4),
                Precision=round(met["precision"], 4),
                Recall=round(met["recall"], 4),
                Accuracy=round(met["accuracy"], 4),
            ))
            for cid in range(NUM_LABELS):
                pc_rows.append(dict(
                    RunDate=DATE_LABEL, Config=cfg["NAME"],
                    Split=split_name, ClassID=cid,
                    ClassName=ID2LABEL[cid],
                    Precision=round(met["per_class_prec"][cid], 4),
                    Recall=round(met["per_class_rec"][cid], 4),
                    F1=round(met["per_class_f1"][cid], 4),
                    Support=sum(1 for y in y_true if y == cid),
                ))

        # JSON metrics
        json_path = f"{out_dir}/metrics_{RUN_DATE}.json"
        with open(json_path, "w") as f:
            json.dump(dict(
                config=cfg["NAME"],
                num_last_layers=NUM_LAST_LAYERS,
                run_date=DATE_LABEL, model=MODEL_NAME,
                val_macro_f1=vm["macro_f1"],
                test_macro_f1=tm["macro_f1"],
                test_accuracy=tm["accuracy"],
                peak_vram_gb=peak_gb,
                runtime_min=elapsed / 60,
                train_loss=train_loss,
            ), f, indent=2)

        # Excel
        excel_local = f"{out_dir}/results_{cfg['NAME']}_{RUN_DATE}.xlsx"
        save_excel(summary_rows, pc_rows, lw_rows, lime_rows, arch_rows, excel_local)

        # Save trained model as .pt
        model_save_dir = f"{out_dir}/trained_model_best_{RUN_DATE}"
        os.makedirs(model_save_dir, exist_ok=True)
        trainer.save_model(model_save_dir)  # saves pytorch_model.pt
        print(f"  Trained model saved (as .pt) → {model_save_dir}")

        # Copy artefacts to Drive
        copy_map = {
            csv_path: f"{drv_dir}/predictions_test_{RUN_DATE}.csv",
            json_path: f"{drv_dir}/metrics_{RUN_DATE}.json",
            cm_path: f"{drv_dir}/confusion_matrix_test_{RUN_DATE}.png",
            excel_local: f"{drv_dir}/results_{cfg['NAME']}_{RUN_DATE}.xlsx",
            f"{out_dir}/layer_weights_{RUN_DATE}.npy": f"{drv_dir}/layer_weights_{RUN_DATE}.npy",
        }
        for src, dst in copy_map.items():
            if os.path.exists(src):
                shutil.copy(src, dst)

        if os.path.isdir(lime_dir):
            shutil.copytree(lime_dir, f"{drv_dir}/lime_figures", dirs_exist_ok=True)
        if os.path.isdir(model_save_dir):
            shutil.copytree(
                model_save_dir,
                f"{drv_dir}/trained_model_best_{RUN_DATE}",
                dirs_exist_ok=True,
            )
        shutil.copy(excel_local, f"{DRIVE_BASE}/results_{cfg['NAME']}_{RUN_DATE}.xlsx")
        print(f"  ✓ All artefacts → {drv_dir}")
        print(f"  ✓ Trained model (.pt) on Drive → {drv_dir}/trained_model_best_{RUN_DATE}")

        return dict(
            val_met=vm, test_met=tm,
            test_f1=tm["macro_f1"], test_acc=tm["accuracy"],
            runtime=elapsed / 60, vram=peak_gb,
            lime_rows=len(lime_rows),
        )

    except Exception as exc:
        import traceback
        print("\n" + "═" * 60 + f"\nERROR: {exc}\n" + "═" * 60)
        traceback.print_exc()
        return {}
    finally:
        for obj in [trainer, train_ds, val_ds, model, tok]:
            try:
                if obj is not None:
                    del obj
            except Exception:
                pass
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  VRAM freed | {cfg['NAME']} done")


# ════════════════════════════════════════════════════════
# CELL 15 — Execute
# ════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print(f"  QWEN3-4B | UNSLOTH | LAST {NUM_LAST_LAYERS} LAYERS | CFG_A HYPERPARAMS")
print(f"  Layer indices: {LAYER_INDICES}")
print("═" * 70)

result = run(CFG)

if result:
    print("\n" + "═" * 70)
    print(f"  FINAL RESULTS — Last {NUM_LAST_LAYERS} Layers [{DATE_LABEL}]")
    print("═" * 70)
    print(f"  Val  Macro F1 : {result['val_met']['macro_f1']:.4f}")
    print(f"  Test Macro F1 : {result['test_met']['macro_f1']:.4f}")
    print(f"  Test Accuracy : {result['test_acc']:.4f}")
    print(f"  Runtime       : {result['runtime']:.1f} min")
    print(f"  Peak VRAM     : {result['vram']:.2f} GB")
    print(f"  LIME rows     : {result['lime_rows']}")
    print("\n  Per-class F1 (Test):")
    for i, lbl in ID2LABEL.items():
        print(f"    [{i}] {lbl:<28} {result['test_met']['per_class_f1'][i]:.4f}")
    print(f"\n  Drive root: {DRIVE_BASE}")
print("\n✓ Run complete.")

✓ All dependencies installed
Enter HF token (hf_...): ··········
✓ HuggingFace login OK
Auto-generated LAYER_INDICES for last 8 layers: [-1, -2, -3, -4, -5, -6, -7, -8]
Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB
Model   : unsloth/Qwen3-4B (36L hidden=2560)
Date    : 02 Jul 2026 16:09
Layers  : last 8 → [-1, -2, -3, -4, -5, -6, -7, -8]
✓ Global config ready
✓ Config ready
  NUM_LAST_LAYERS = 8
  Layer indices   : [-1, -2, -3, -4, -5, -6, -7, -8]
  LoRA r          : 64
  Head growth     : 256 × 4
  Max epochs      : 6
  Unfreeze blocks : 12
✓ 3 prompt variants ready
✓ Leakage audit passed | Train=1278 | Test=191
✓ Split | Train=1150 | Val=128 | Test=191
✓ Prompt worst-case: 274 tokens (limit=320)
✓ Token audit passed
Mounted at /content/drive
✓ Drive mounted → /content/drive/MyDrive/SemEval2026/Qwen3_4B_Unsloth/20260702_1609

══════════════════════════════════════════════════════════════════════
  QWEN3-4B | UNSLOTH | LAST 8 LAYERS | CFG_A HYPERPARAMS
  Layer indices: [-1, -2, -

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
  hidden=2560 n_layers=36 total_hs=37
  li=  -1 → hidden_states[36] (Layer 35)
  li=  -2 → hidden_states[35] (Layer 34)
  li=  -3 → hidden_states[34] (Layer 33)
  li=  -4 → hidden_states[33] (Layer 32)
  li=  -5 → hidden_states[32] (Layer 31)
  li=  -6 → hidden_states[31] (Layer 30)
  li=  -7 → hidden_states[30] (Layer 29)
  li=  -8 → hidden_states[29] (Layer 28)
  ✓ Unfrozen attention norms in layers [24–35] (61,440 params)
  ✓ Unfrozen lm_head (0 params)
  [DenseResHead] 2560 → bn1024 → dense×4(g=256) → trans512 → 4
  Trainable: 137,303,564 / 2,645,828,620 (5.19%)
  Architecture table: 929 parameter entries
  VRAM after load: 4.97 GB
[2/9] Tokenising datasets …
  train=1150 val=128 test=191
[3/9] spe=71 total_steps≈426 warmup=42
[4/9] Training …


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,150 | Num Epochs = 6 | Total steps = 432
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 137,303,564 of 4,159,710,220 (3.30% trained)


Step,Training Loss,Validation Loss,F1 Macro,F1 Micro,Accuracy
75,0.888125,1.127316,0.729565,0.750000,0.750000
150,0.700728,0.735170,0.787158,0.796875,0.796875
225,0.357522,0.633543,0.864988,0.867188,0.867188
300,0.229212,0.446815,0.913099,0.914062,0.914062
375,0.203697,0.478005,0.897261,0.898438,0.898438
432,0.218128,0.477189,0.897261,0.898438,0.898438


  ✓ Model state_dict saved → /content/qwen3_4b_20260702_1609/CFG_Last8_OrigLoRA_SavePT/checkpoint-75/pytorch_model.pt
  ✓ Model state_dict saved → /content/qwen3_4b_20260702_1609/CFG_Last8_OrigLoRA_SavePT/checkpoint-150/pytorch_model.pt
  ✓ Model state_dict saved → /content/qwen3_4b_20260702_1609/CFG_Last8_OrigLoRA_SavePT/checkpoint-225/pytorch_model.pt
  ✓ Model state_dict saved → /content/qwen3_4b_20260702_1609/CFG_Last8_OrigLoRA_SavePT/checkpoint-300/pytorch_model.pt
  ✓ Model state_dict saved → /content/qwen3_4b_20260702_1609/CFG_Last8_OrigLoRA_SavePT/checkpoint-375/pytorch_model.pt
  ✓ Model state_dict saved → /content/qwen3_4b_20260702_1609/CFG_Last8_OrigLoRA_SavePT/checkpoint-432/pytorch_model.pt
  ← Loading best checkpoint .pt: /content/qwen3_4b_20260702_1609/CFG_Last8_OrigLoRA_SavePT/checkpoint-300/pytorch_model.pt
  Loss=0.4915 Time=78.0min PeakVRAM=6.13GB
[5/9] Validation inference …
  TTA: soft-voting over 3 prompt variants …

═══════════════════════════════════════════════